# 10. Tuning to the junction counts: lanes + friction

**Question:** which lever brings the model's junction volumes to the JICA 2011-12 counts?
**Target:** GEH < 5 at all 6 tuning junctions (Chabahil, Gausala, Kalimati, New Baneshwor, Old Baneshwor, Thapathali), on the mean volume over 5 seeds, with road speeds still near the JICA speed survey (Ring Road 24.0, inner main roads 15.3 km/h).

The other 3 counted junctions (Balaju, Kalanki, Koteshwor) were set aside by a fixed random draw before any tuning run and are not shown here; they are checked only once a configuration has been chosen.

**This notebook:** Rounds 1-4 always left the Ring Road junctions (Chabahil, Gausala) and Kalimati too high and the Baneshwor / Thapathali junctions too low. A network check found why: UXsim's OpenStreetMap importer reads the `lanes` tag as lanes per direction, but on a two-way road OpenStreetMap counts both directions, so every tagged two-way road (the Ring Road among them) had twice its width, while the one-way Baneshwor and Thapathali roads were right. Here every tagged two-way road gets half its lanes per direction, and the 10% of main road links with the most shops, amenities and offices within 100 m (from OpenStreetMap) get a 30 km/h free-flow speed instead of 60, for stopping buses, pedestrians and street trade. The rules apply to the whole city and were fixed before any run; they do not look at the counts. Trips load at mid-block nodes as in round 4. Every configuration runs on seeds 1-5, next to the unchanged baseline. Volumes are measured in the second hour of a two-hour steady peak.

In [ ]:
import glob, os, shutil, subprocess, sys, zipfile

def find_project(marker):
    """Copy the project tree that contains `marker` from /kaggle/input to /kaggle/working/sim."""
    hits = sorted(glob.glob(f"/kaggle/input/**/{marker}", recursive=True),
                  key=lambda h: ("build-demand" not in h, len(h)))   # prefer step 1's copy
    if not hits:                                        # dataset delivered as a zip
        for z in glob.glob("/kaggle/input/**/*.zip", recursive=True):
            zipfile.ZipFile(z).extractall("/kaggle/working/_unzipped")
        hits = sorted(glob.glob(f"/kaggle/working/_unzipped/**/{marker}", recursive=True), key=len)
    src = hits[0][: -len(marker)].rstrip("/")
    shutil.copytree(src, "/kaggle/working/sim", dirs_exist_ok=True)
    os.chdir("/kaggle/working/sim")
    print("project from", src)

def run(*cmd, tries=1):
    """Run a script, streaming its output; retry resumable drivers if a worker is killed."""
    for attempt in range(tries):
        p = subprocess.Popen([sys.executable, "-u", *cmd], stdout=subprocess.PIPE,
                             stderr=subprocess.STDOUT, text=True)
        for line in p.stdout:
            print(line, end="")
        if p.wait() == 0:
            return
        print(f"exit code {p.returncode}, attempt {attempt + 1} of {tries}")
    raise RuntimeError(" ".join(cmd))

find_project("scripts/tune_counts.py")

In [ ]:
!pip install -q uxsim==1.14.2

In [ ]:
run("scripts/build_minor_ways.py")
run("scripts/build_od_jica.py", "--data", "data/jica_net")

In [ ]:
run("scripts/tune_counts.py", "--idea", "osmlanes_friction", "--seeds", "5", "--jobs", "3", tries=3)

In [ ]:
run("scripts/tune_counts.py", "--idea", "osmlanes_friction", "--summary-only", "--radius", "100")